# The reference assembly: satellites and centromeres of T2T-goat2.0
This notebook takes me from the raw genome downloads to a table with the centromere of every chromosome. It has two sections, first the data and then the satellites and centromeres. Everything runs on my laptop, and every table I produce is saved in `data/`.

| Section | What I do | Main output |
|---|---|---|
| 1 · Data and assemblies | Download the T2T-goat2.0 genomes (NCBI) and PacBio reads of other breeds (ENA), and standardise names | `chromosomes.csv`, `manifest.csv` |
| 2 · Satellites and centromeres | Orient each chromosome, find the satellite units with SRF, annotate them and delimit each centromere | `centromeres_master.csv` |

I start from T2T-goat2.0 (Wu et al. 2024, *Nat Commun* 15:10041) because, out of the 70 *Capra hircus* assemblies in NCBI, it is the only one whose authors resolved all the centromeres. It is the reference of the whole project: the satellite units I find here are the yardstick I use in notebook 02 to measure the other assemblies, and the statistics are in notebook 03. The genome belongs to a male Inner Mongolia Cashmere goat, `IMCG_204112`. I use the collapsed assembly (`GCA_041735815.1`) and its two haplotypes, P (`GCA_041053135.1`) and M (`GCA_040970035.1`). The PacBio reads of other breeds come from BioProject `PRJNA1062519`.

Two decisions shape everything. I tried CENP-A ChIP-seq and dropped it (the reason is in `docs/NOTE_chipseq_removed.Rmd`), so I define a centromere by how much satellite DNA a block contains. And I treat the comparison between breeds as a side, exploratory question (notebook 03, section 3.4).

A few things to keep in mind when reading the results. The three assemblies come from one animal, so P and M are two haplotypes and not two individuals. My centromere is a satellite-rich block and not the CENP-A domain, so it can be longer than the 0.6 to 8.51 Mb that Wu et al. report. SatIII is a short 22-bp motif that aligns poorly, so I probably underestimate it. The X has its satellite block about 7.2 Mb from the end because its short arm carries the pseudoautosomal region. I check in section 2.7 that this is not an error, but it still weighs a lot in the statistics. And three centromeres of haplotype M have very little satellite, which looks like missing sequence in the assembly. I flag them as `low_satellite` and repeat the statistics without them.

To run it, open it from the repository root (or from `notebooks/`) with the packages in `requirements.txt`. minimap2 and samtools come from Homebrew, and cell 1.2 installs KMC, SRF and k8 in `tools/`.

# Section 1 · Data and assemblies

## 1.1 · Configuration

In [1]:
import os, sys, re, gzip, hashlib, subprocess, shutil, json, io, time, shlex, platform
import urllib.request
from pathlib import Path
import numpy as np, pandas as pd, requests

def urlopen_retry(url, tries=6):           # intermittent DNS/network failures: retry with increasing wait
    for k in range(tries):
        try: return urllib.request.urlopen(url, timeout=60)
        except Exception as e:
            if k == tries-1: raise
            time.sleep(10*(k+1))
Q = lambda p: shlex.quote(str(p))          # paths with spaces (e.g. 'Capra hircus') in shell commands

# ---------- parameters ----------
DOWNLOAD_READS  = True      # PacBio reads (subsampled); False to skip them
PACBIO_BASES_GB = 0.5       # Gb of bases to keep per run (~0.17x per breed; ENA delivers ~0.5 MB/s)
WINDOW_MB       = 15        # proximal window per chromosome (centromeres span 0.6–10 Mb)
MAX_GAP         = 500_000   # bp; maximum separation between satellite blocks of the same centromere
MIN_SAT_FRAC    = 0.30      # a block with less satellite than this fraction is flagged low_satellite
KINDS_SAT       = ['SatI','SatII','SatIII']
SEED            = 20241013
THREADS         = os.cpu_count() or 2

# ---------- paths (all relative to the project folder) ----------
BASE  = Path.cwd()
if BASE.name == 'notebooks': BASE = BASE.parent        # allow running from notebooks/
DATA  = BASE/'data'; TAB = DATA                         # tables, raw downloads and reads
RAW, READS, FIG = DATA/'raw_ncbi', DATA/'reads', DATA/'figures'
ROOT  = BASE/'work'; CLEAN = ROOT/'clean'               # renamed FASTA and windows (regenerated in minutes)
TOOLS = BASE/'tools'
for d in (RAW, READS, FIG, CLEAN, TOOLS): d.mkdir(parents=True, exist_ok=True)

# ---------- sources ----------
ASSEMBLIES = {   # label -> (GenBank accession, NCBI name)
    'T2T': ('GCA_041735815.1', 'T2T-goat2.0'),    # collapsed (30 chromosomes + Y)
    'P'  : ('GCA_041053135.1', 'T2T-goat2.0P'),   # paternal haplotype (autosomes + Y)
    'M'  : ('GCA_040970035.1', 'T2T-goat2.0M'),   # maternal haplotype (autosomes + X)
}
ENA_PROJECT = 'PRJNA1062519'   # NCBI/ENA; CNCB mirror: PRJCA022847
print('Project:', BASE); print('Tables :', TAB)

Project: <repo>
Tables : <repo>/data


## 1.2 · Tools

In [2]:
# minimap2 and samtools: Homebrew (brew install minimap2 samtools). KMC, SRF and k8 are installed in ./tools (macOS arm64).
for t in ('minimap2', 'samtools'):
    assert shutil.which(t), f'missing {t}: brew install {t}'
KMC, KMC_DUMP = TOOLS/'bin'/'kmc', TOOLS/'bin'/'kmc_dump'
SRF, SRFUTILS, K8 = TOOLS/'srf'/'srf', TOOLS/'srf'/'srfutils.js', TOOLS/'k8'
if not all(p.exists() for p in (KMC, KMC_DUMP, SRF, SRFUTILS, K8)):
    assert platform.system() == 'Darwin' and platform.machine() == 'arm64', 'automatic installation only for macOS arm64'
    script = f'''
set -e; cd {Q(TOOLS)}
curl -sL -o kmc.tgz https://github.com/refresh-bio/KMC/releases/download/v3.2.4/KMC3.2.4.mac.arm64.tar.gz && tar xzf kmc.tgz
mkdir -p srf && curl -sL https://github.com/lh3/srf/archive/refs/heads/master.tar.gz | tar xz -C srf --strip-components=1 && make -C srf > /dev/null
curl -sL -o k8.tbz https://github.com/attractivechaos/k8/releases/download/v1.2/k8-1.2.tar.bz2 && tar xjf k8.tbz && cp k8-1.2/k8-arm64-Darwin k8 && chmod +x k8
rm -rf kmc.tgz k8.tbz k8-1.2
'''
    subprocess.run(['bash', '-c', script], check=True)
for p in (KMC, KMC_DUMP, SRF, SRFUTILS, K8): assert p.exists(), p
print('Tools OK:', {t: shutil.which(t) for t in ('minimap2', 'samtools')}, '+ KMC, SRF, k8 in', TOOLS)

Tools OK: {'minimap2': '/opt/homebrew/bin/minimap2', 'samtools': '/opt/homebrew/bin/samtools'} + KMC, SRF, k8 in <repo>/tools


## 1.3 · Downloading the assemblies (NCBI)
Each assembly is about 0.86 GB. I check every file against its MD5, and if it is already there and intact I do not download it again.

In [3]:
def ncbi_dir(acc, name):
    n = acc.split('_')[1].split('.')[0]
    return f'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/{n[:3]}/{n[3:6]}/{n[6:9]}/{acc}_{name}/'

def md5_file(p, chunk=1<<24):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        while b := f.read(chunk): h.update(b)
    return h.hexdigest()

def fetch(url, dest):
    subprocess.run(['curl', '-sS', '-L', '--retry', '5', '-C', '-', '-o', str(dest), url], check=True)

manifest = []   # (type, id, file, bytes, md5, source)
for hap, (acc, name) in ASSEMBLIES.items():
    base = ncbi_dir(acc, name); stem = f'{acc}_{name}'
    md5s = {l.split()[1].lstrip('./'): l.split()[0] for l in urlopen_retry(base + 'md5checksums.txt').read().decode().splitlines() if l.strip()}
    fa_gz = RAW/(stem + '_genomic.fna.gz')
    if not (fa_gz.exists() and md5_file(fa_gz) == md5s[stem + '_genomic.fna.gz']):   # already downloaded and intact: not repeated
        fetch(base + stem + '_genomic.fna.gz', fa_gz)
    fetch(base + stem + '_assembly_report.txt', RAW/(stem + '_assembly_report.txt'))
    got = md5_file(fa_gz)
    assert got == md5s[stem + '_genomic.fna.gz'], f'MD5 mismatch in {hap}'
    manifest.append(('assembly', hap, stem + '_genomic.fna.gz', (RAW/(stem+'_genomic.fna.gz')).stat().st_size, got, base))
    print(hap, name, 'OK  MD5', got)

T2T T2T-goat2.0 OK  MD5 87c5212a7acb095d1d9358c67e3c952e


P T2T-goat2.0P OK  MD5 9e4880b5d6ecd19ac677b400f17eaec6


M T2T-goat2.0M OK  MD5 b5c25a68070fff3eec4f5258a2534546


## 1.4 · Chromosome table and renaming
I take the names from the NCBI sequence report and standardise them to `CHI1…CHI29`, `CHIX` and `CHIY`. In 2.0M the X appears as `30`, but its length (144.69 Mb) is the same as the X of T2T-goat2.0, so I rename it by length. Each haplotype has 30 chromosomes: 29 autosomes plus one sex chromosome, the Y in P and the X in M. I look for their gaps in section 2.

In [4]:
import pandas as pd
def sequence_report(acc):
    rows, token = [], None
    while True:
        u = f'https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{acc}/sequence_reports?page_size=1000' + (f'&page_token={token}' if token else '')
        d = json.load(urlopen_retry(u)); rows += d['reports']; token = d.get('next_page_token')
        if not token: return rows

chrom_rows = []
for hap, (acc, name) in ASSEMBLIES.items():
    for r in sequence_report(acc):
        chrom_rows.append(dict(hap=hap, assembly=name, accession=r['genbank_accession'], ncbi_name=r.get('chr_name'),
                               role=r.get('role'), length=r['length'], gc_percent=r.get('gc_percent')))
chroms = pd.DataFrame(chrom_rows)
chroms['placed'] = chroms.role.eq('assembled-molecule') & chroms.ncbi_name.notna()
# 2.0M names the X as '30': detected by its length, identical to the X of T2T-goat2.0
x_len = chroms[(chroms.hap=='T2T') & (chroms.ncbi_name=='X')].length.iloc[0]
fix = chroms.placed & chroms.ncbi_name.str.fullmatch(r'\d+').fillna(False) & (chroms.length==x_len)
chroms.loc[fix, 'ncbi_name'] = 'X'
chroms['chr'] = chroms.apply(lambda r: f'CHI{r.ncbi_name}' if r.placed else None, axis=1)
chroms['seqname'] = chroms.apply(lambda r: f'{r.hap}_{r.chr}' if r.placed else f'{r.hap}_unplaced_{r.accession}', axis=1)
chroms['kind'] = chroms.chr.map(lambda c: None if c is None else ('X' if c=='CHIX' else 'Y' if c=='CHIY' else 'autosome'))

print(chroms.groupby(['hap','placed']).agg(n=('length','size'), Mb=('length', lambda s: round(s.sum()/1e6,1))).unstack())
# expected checks according to the paper
for hap, g in chroms[chroms.placed].groupby('hap'):
    au = (g.kind=='autosome').sum()
    print(hap, 'autosomes:', au, '| X:', (g.kind=='X').sum(), '| Y:', (g.kind=='Y').sum())
    assert au == 29, 'expected 29 autosomes'
    assert (g.kind=='X').sum() == (hap in ('T2T','M')) and (g.kind=='Y').sum() == (hap in ('T2T','P')), 'unexpected X/Y'
chroms.to_csv(TAB/'chromosomes.csv', index=False)
chroms[chroms.placed].head(3)

          n      Mb
placed True    True
hap                
M        30  2805.1
P        30  2709.9
T2T      31  2863.9
M autosomes: 29 | X: 1 | Y: 0
P autosomes: 29 | X: 0 | Y: 1
T2T autosomes: 29 | X: 1 | Y: 1


,hap,assembly,accession,ncbi_name,role,length,gc_percent,placed,chr,seqname,kind
0,T2T,T2T-goat2.0,CP168611.1,1,assembled-molecule,166528736,41.0,True,CHI1,T2T_CHI1,autosome
1,T2T,T2T-goat2.0,CP168612.1,2,assembled-molecule,147555794,42.0,True,CHI2,T2T_CHI2,autosome
2,T2T,T2T-goat2.0,CP168613.1,3,assembled-molecule,132669774,43.0,True,CHI3,T2T_CHI3,autosome


In [5]:
# Rename FASTA records: >{hap}_CHI{n}; unplaced sequences keep their accession. Indexed with samtools faidx.
def rename_fasta(hap):
    acc, name = ASSEMBLIES[hap]
    mp = chroms[chroms.hap==hap].set_index('accession').seqname.to_dict()
    src = RAW/f'{acc}_{name}_genomic.fna.gz'; out = CLEAN/f'{hap}.fa'
    if Path(str(out)+'.fai').exists(): return out      # already generated
    seen = set()
    with gzip.open(src, 'rt') as fi, open(out, 'w') as fo:
        for line in fi:
            if line.startswith('>'):
                a = line[1:].split()[0]; fo.write(f'>{mp[a]}\n'); seen.add(a)
            else:
                fo.write(line.upper())          # no soft-masking: simplifies N and k-mer counting
    assert seen == set(mp), f'{hap}: missing sequences {set(mp)-seen}'
    subprocess.run(['samtools','faidx',str(out)], check=True)
    return out
GENOME = {h: rename_fasta(h) for h in ASSEMBLIES}
for h,p in GENOME.items(): print(h, p, round(p.stat().st_size/1e9,2), 'GB')

T2T <repo>/work/clean/T2T.fa 2.9 GB
P <repo>/work/clean/P.fa 2.74 GB
M <repo>/work/clean/M.fa 2.84 GB


## 1.5 · PacBio reads from other breeds
I also take reads from Boer, Tibetan, Jining gray, Zhongwei and Liaoning goats, and from `IMCG_204112` itself, to compare how much satellite each animal carries (notebook 03, section 3.4). They are noisy CLR subreads, good enough to estimate satellite fractions but not to resolve arrays. I stream each file and stop after 0.5 Gb, because the complete runs are 50 to 120 GB each.

In [6]:
ena = pd.read_csv(f'https://www.ebi.ac.uk/ena/portal/api/filereport?accession={ENA_PROJECT}&result=read_run&fields=run_accession,sample_title,experiment_title,library_strategy,library_name,instrument_platform,read_count,base_count,fastq_ftp,fastq_bytes&format=tsv', sep='\t')
# Cleaning: only PacBio WGS runs with FASTQ files; RNA-seq, ChIP-seq and runs without FASTQ are discarded
runs = ena[ena.fastq_ftp.notna() & (ena.instrument_platform=='PACBIO_SMRT') & (ena.library_strategy=='WGS')].copy()
runs['type'] = 'PacBio'
runs['individual'] = runs.sample_title.str.replace(' for PacBio sequencing','',regex=False)
runs['Gb'] = runs.fastq_bytes.astype(str).str.split(';').map(lambda x: sum(map(float,x))/1e9)
runs = runs[['run_accession','type','individual','instrument_platform','library_strategy','Gb','fastq_ftp']].sort_values(['type','individual']).reset_index(drop=True)
runs.to_csv(TAB/'public_runs.csv', index=False)
display(runs.drop(columns='fastq_ftp'))

,run_accession,type,individual,instrument_platform,library_strategy,Gb
0,SRR29611984,PacBio,Boer goat,PACBIO_SMRT,WGS,52.715141
1,SRR29612035,PacBio,IMCG_204112,PACBIO_SMRT,WGS,117.772901
2,SRR29611983,PacBio,Jining gray goat,PACBIO_SMRT,WGS,75.160629
3,SRR29611982,PacBio,Liaoning cashmere goat,PACBIO_SMRT,WGS,76.010371
4,SRR29612031,PacBio,Tibetan goat,PACBIO_SMRT,WGS,80.914748
5,SRR29612034,PacBio,Zhongwei goat,PACBIO_SMRT,WGS,69.862471


In [7]:
def stream_lines(url):
    r = requests.get(url, stream=True, timeout=60); r.raise_for_status()
    return gzip.GzipFile(fileobj=r.raw)

def subsample_fastq_bases(url, out, max_bases):
    s, tot, n = stream_lines(url), 0, 0
    with gzip.open(str(out) + '.part','wb') as o:
        while tot < max_bases:
            rec = [s.readline() for _ in range(4)]
            if not rec[0]: break
            o.writelines(rec); tot += len(rec[1].strip()); n += 1
    os.replace(str(out) + '.part', out)          # published only when finished: no truncated files
    return n, tot

from concurrent.futures import ThreadPoolExecutor
def with_retry(fn, *a, tries=5):
    for k in range(tries):
        try: return fn(*a)
        except Exception as e:
            print('  retry', k+1, type(e).__name__); time.sleep(5*(k+1))
    raise RuntimeError('download failed after %d attempts' % tries)

def get_run(r):
    url = 'https://' + r.fastq_ftp.split(';')[0]
    out = READS/f'{r.run_accession}_{re.sub(r"[^A-Za-z0-9]+", "_", r.individual)}.subreads.fastq.gz'
    if not out.exists(): print(r.run_accession, r.individual, with_retry(subsample_fastq_bases, url, out, int(PACBIO_BASES_GB*1e9)))
    return r, [out]

if DOWNLOAD_READS:
    # ENA limits ~0.5 MB/s per connection: 4 runs are downloaded in parallel
    with ThreadPoolExecutor(4) as ex:
        for r, files in ex.map(get_run, list(runs.itertuples())):
            for f in files:
                manifest.append(('reads', r.run_accession, f.name, f.stat().st_size, md5_file(f), 'https://www.ebi.ac.uk/ena/browser/view/' + r.run_accession))
else:
    print('DOWNLOAD_READS=False: raw reads are skipped')

## 1.6 · Download manifest

In [8]:
man = pd.DataFrame(manifest, columns=['type','id','file','bytes','md5','source'])
man['download_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
man.to_csv(TAB/'manifest.csv', index=False)
print(man[['type','id','file']].assign(MB=(man.bytes/1e6).round(1)).to_string(index=False))

    type          id                                                 file    MB
assembly         T2T           GCA_041735815.1_T2T-goat2.0_genomic.fna.gz 864.7
assembly           P          GCA_041053135.1_T2T-goat2.0P_genomic.fna.gz 817.6
assembly           M          GCA_040970035.1_T2T-goat2.0M_genomic.fna.gz 852.6
   reads SRR29611984              SRR29611984_Boer_goat.subreads.fastq.gz 309.7
   reads SRR29612035            SRR29612035_IMCG_204112.subreads.fastq.gz 405.6
   reads SRR29611983       SRR29611983_Jining_gray_goat.subreads.fastq.gz 396.0
   reads SRR29611982 SRR29611982_Liaoning_cashmere_goat.subreads.fastq.gz 366.6
   reads SRR29612031           SRR29612031_Tibetan_goat.subreads.fastq.gz 149.7
   reads SRR29612034          SRR29612034_Zhongwei_goat.subreads.fastq.gz 359.1


# Section 2 · Satellites and centromeres

## 2.1 · End windows and gaps
Goat autosomes are acrocentric, so the centromere sits near one end. I extract the first and last 15 Mb of every chromosome and mark the runs of N as gaps. I leave the Y out of the satellite analysis, as the paper does, because it has its own centromere.

In [9]:
from pyfaidx import Fasta
W = WINDOW_MB*1_000_000
def gap_runs(seq, min_len=1):
    return [(m.start(), m.end()) for m in re.finditer(r'N{%d,}' % min_len, seq)]

win_rows, gap_rows = [], []
for hap, path in GENOME.items():
    fa = Fasta(str(path), as_raw=True, sequence_always_upper=True)
    with open(CLEAN/f'{hap}.left{WINDOW_MB}Mb.fa','w') as fl, open(CLEAN/f'{hap}.right{WINDOW_MB}Mb.fa','w') as fr:
        for r in chroms[(chroms.hap==hap) & chroms.placed].itertuples():
            L = r.length; w = min(W, L)
            left, right = fa[r.seqname][0:w], fa[r.seqname][L-w:L]
            fl.write(f'>{r.seqname}:L\n{left}\n'); fr.write(f'>{r.seqname}:R\n{right}\n')
            for s,e in gap_runs(left): gap_rows.append(dict(hap=hap, chr=r.chr, kind=r.kind, start=s, end=e, len=e-s))
            win_rows.append(dict(hap=hap, chr=r.chr, kind=r.kind, length=L,
                                 N_left=left.count('N'), N_right=right.count('N')))
windows = pd.DataFrame(win_rows); gaps = pd.DataFrame(gap_rows)
chrom_qc = windows.merge(gaps.groupby(['hap','chr']).size().rename('gaps_prox').reset_index(), how='left', on=['hap','chr']).fillna({'gaps_prox':0})
print('Chromosomes with gaps in the proximal window (autosomes + X), by haplotype:')
print(chrom_qc[(chrom_qc.gaps_prox>0)&(chrom_qc.kind!='Y')].groupby('hap').chr.apply(list).to_string())
chrom_qc.to_csv(TAB/'qc_windows.csv', index=False)

Chromosomes with gaps in the proximal window (autosomes + X), by haplotype:
hap
M            [CHI2, CHI10, CHI12, CHI20, CHI22, CHI23]
P    [CHI2, CHI4, CHI10, CHI11, CHI17, CHI23, CHI27...


## 2.2 · Satellite reference and a first guess of the orientation
SatI comes from the sequence `X57335.1` (*C. hircus* satellite 1.714, 816 bp), the one Wu et al. cite. I align it to both ends of each chromosome to guess which end holds the centromere. That reference is short and diverged, so for most chromosomes it hits nothing on either side and the guess defaults to the left end. For that reason I check the orientation again in 2.4 with all the SRF units.

In [10]:
def efetch_fasta(acc):
    u = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={acc}&rettype=fasta'
    t = urlopen_retry(u).read().decode().strip().splitlines()
    return ''.join(t[1:]).upper()
satI = efetch_fasta('X57335.1'); assert len(satI) == 816, len(satI)
SATREF = TAB/'SatI_X57335.fa'; SATREF.write_text(f'>SatI_X57335.1\n{satI}\n')
print('SatI', len(satI), 'bp  ->', SATREF)

SatI 816 bp  -> <repo>/data/SatI_X57335.fa


In [11]:
# Orientation: SatI coverage at the left vs right end (acrocentric chromosomes, no Y)
import numpy as np
def sati_coverage(fa_path, out_paf):
    with open(out_paf,'w') as fo:
        subprocess.run(['minimap2','-x','asm20','-N','1000000','-f','10000000','-t',str(THREADS),str(fa_path),str(SATREF)], stdout=fo, stderr=subprocess.DEVNULL, check=True)
    cols = ['q','ql','qs','qe','strand','t','tl','ts','te','nm','al','mq']
    df = pd.read_csv(out_paf, sep='\t', header=None, usecols=range(12), names=cols)
    cov = {}
    for t, g in df.groupby('t'):
        iv = sorted(zip(g.ts, g.te)); tot, cs, ce = 0, *iv[0]
        for s,e in iv[1:]:
            if s > ce: tot += ce-cs; cs, ce = s, e
            else: ce = max(ce, e)
        cov[t] = tot + ce - cs
    return cov

orient = []
for hap in GENOME:
    cL = sati_coverage(CLEAN/f'{hap}.left{WINDOW_MB}Mb.fa',  CLEAN/f'{hap}.L.paf')
    cR = sati_coverage(CLEAN/f'{hap}.right{WINDOW_MB}Mb.fa', CLEAN/f'{hap}.R.paf')
    for r in chroms[(chroms.hap==hap)&chroms.placed&(chroms.kind!='Y')].itertuples():
        orient.append(dict(hap=hap, chr=r.chr, satI_left_Mb=cL.get(f'{r.seqname}:L',0)/1e6, satI_right_Mb=cR.get(f'{r.seqname}:R',0)/1e6))
orient = pd.DataFrame(orient)
orient['centromere_on_left'] = orient.satI_left_Mb >= orient.satI_right_Mb      # a tie (no signal) goes to the left
orient['satI_signal'] = (orient.satI_left_Mb + orient.satI_right_Mb) >= 0.01   # Mb; below this the reference says nothing
print('Chromosomes where the SatI reference gives a usable signal, by haplotype:'); print(orient.groupby('hap').satI_signal.agg(['sum','size']))
print('Among those, more SatI on the right:', orient[orient.satI_signal & ~orient.centromere_on_left][['hap','chr']].values.tolist())
orient.to_csv(TAB/'qc_orientation.csv', index=False)

Chromosomes where the SatI reference gives a usable signal, by haplotype:
     sum  size
hap           
M      0    30
P      1    29
T2T    0    30
Among those, more SatI on the right: []


In [12]:
from Bio.Seq import Seq
side = {(r.hap, r.chr): ('L' if r.satI_left_Mb >= r.satI_right_Mb else 'R') for r in orient.itertuples()}
orient['centromere_side'] = [side[(r.hap, r.chr)] for r in orient.itertuples()]
orient.to_csv(TAB/'qc_orientation.csv', index=False)
print('End carrying the centromere (SatI), by haplotype:'); print(orient.groupby(['hap','centromere_side']).size().unstack(fill_value=0))
PROX = {}
gap_rows = []
for hap, path in GENOME.items():
    fa = Fasta(str(path), as_raw=True, sequence_always_upper=True)
    PROX[hap] = CLEAN/f'{hap}.prox{WINDOW_MB}Mb.fa'
    with open(PROX[hap], 'w') as fo:
        for r in chroms[(chroms.hap==hap) & chroms.placed & (chroms.kind!='Y')].itertuples():
            w = min(W, r.length)
            s = fa[r.seqname][0:w] if side[(hap, r.chr)] == 'L' else str(Seq(fa[r.seqname][r.length-w:r.length]).reverse_complement())
            fo.write(f'>{r.seqname}\n{s}\n')
            for a, b in gap_runs(s): gap_rows.append(dict(hap=hap, chr=r.chr, kind=r.kind, start=a, end=b, len=b-a))
gaps = pd.DataFrame(gap_rows, columns=['hap','chr','kind','start','end','len']); gaps.to_csv(TAB/'proximal_gaps.csv', index=False)
print('Gaps in the oriented proximal window:', gaps.groupby('hap').chr.nunique().to_dict(), 'chromosomes with gaps')

End carrying the centromere (SatI), by haplotype:
centromere_side   L  R
hap                   
M                30  0
P                28  1
T2T              30  0


Gaps in the oriented proximal window: {'M': 6, 'P': 9} chromosomes with gaps


## 2.3 · Satellite units with SRF
SRF (Satellite Repeat Finder, by Heng Li) counts the frequent 151-mers with KMC and assembles them into repeat units. For higher-order repeats it returns several copies in a row. I run it on the first 10 Mb of the proximal window of T2T-goat2.0, as in `srf.sh` from the authors' repository.

In [13]:
# SRF on the oriented proximal windows (T2T-goat2.0, autosomes + X) — reproduces centromere_unit_identification/srf.sh
SRFDIR = CLEAN/'srf'; SRFDIR.mkdir(exist_ok=True)
SRF_WINDOW = 10_000_000     # SRF only needs the satellite block
KMC_RAM_GB = 12
STEP_TIMEOUT_MIN = 30       # if a step exceeds this, it is cut and reported (instead of hanging)
fa = Fasta(str(PROX['T2T']), as_raw=True, sequence_always_upper=True)
with open(SRFDIR/'prox.fa','w') as fo:
    for name in fa.keys():
        fo.write(f'>{name.split("_",1)[1]}\n{fa[name][0:SRF_WINDOW]}\n')

steps = [
 ('kmc',   f"{Q(KMC)} -fm -k151 -t{min(THREADS,8)} -m{KMC_RAM_GB} -sm -ci20 -cs1000000 prox.fa count.kmc tmp_dir > /dev/null"),
 ('dump',  f"{Q(KMC_DUMP)} count.kmc count.txt"),
 ('srf',   f"{Q(SRF)} -p prefix count.txt > srf.fa"),
]
subprocess.run(f"cd {Q(SRFDIR)}; rm -rf tmp_dir count.kmc*; mkdir -p tmp_dir", shell=True, check=True)
for name, cmd in steps:
    t0 = time.time()
    try:
        subprocess.run(['bash','-o','pipefail','-c', f'cd {Q(SRFDIR)}; {cmd}'], check=True, timeout=STEP_TIMEOUT_MIN*60)
    except subprocess.TimeoutExpired:
        raise SystemExit(f'Step {name} exceeded {STEP_TIMEOUT_MIN} min; check the size of count.txt / RAM')
    extra = {p: (SRFDIR/p).stat().st_size for p in ('count.txt','srf.fa') if (SRFDIR/p).exists()} if name in ('dump','srf') else ''
    print(f'{name:6s} {time.time()-t0:6.0f} s', extra)
print('SRF units:', sum(1 for l in open(SRFDIR/'srf.fa') if l.startswith('>')))
print(subprocess.check_output(f"grep '>' {Q(SRFDIR/'srf.fa')} | head -30", shell=True, text=True))

********************

*******
Stage 1: 100%


Stage 2: 95%

Stage 2: 98%

Stage 2: 100%


kmc         5 s 
dump        0 s {'count.txt': 172801142, 'srf.fa': 28997}


[M::main] read 1114155 distinct k-mers


srf         1 s {'count.txt': 172801142, 'srf.fa': 28997}
SRF units: 21
>prefix#circ1-6 min=35912,max=36104,avg=35979
>prefix#circ2-1406 min=2763,max=10644,avg=5413
>prefix#circ3-6 min=2332,max=2339,avg=2336
>prefix#circ4-132 min=617,max=1513,avg=919
>prefix#circ5-132 min=177,max=1166,avg=490
>prefix#circ6-2115 min=31,max=798,avg=117
>prefix#circ7-3235 min=32,max=732,avg=114
>prefix#circ8-770 min=26,max=531,avg=116
>prefix#circ9-4067 min=91,max=336,avg=198
>prefix#circ10-816 min=43,max=259,avg=88
>prefix#circ11-4080 min=38,max=230,avg=99
>prefix#circ12-770 min=23,max=171,avg=61
>prefix#circ13-2043 min=41,max=161,avg=124
>prefix#circ14-2459 min=24,max=144,avg=74
>prefix#circ15-2445 min=31,max=116,avg=78
>prefix#circ16-3262 min=25,max=93,avg=54
>prefix#circ17-132 min=20,max=89,avg=65
>prefix#circ18-44 min=34,max=75,avg=52
>prefix#circ19-52 min=34,max=36,avg=35
>prefix#circ20-77 min=21,max=34,avg=22
>prefix#circ21-50 min=20,max=20,avg=20



### Classifying the units
I classify each unit by its sequence, not only by its length. If the 22-bp SatIII motif (`GGAAATCACGTGGGCCCCCAGG` or its variant `GGCAATCACG…`) covers at least 60 % of a unit, the unit is SatIII. This includes the 44- and 132-bp units and also larger ones (770 and 2115 bp) that turn out to be mostly motif repeats. Otherwise, a length that is a multiple of 816 bp makes the unit SatI, and a multiple of 702 bp makes it SatII, both with 4 % tolerance because SRF returns higher-order repeats. Everything else, such as 6-bp microsatellites, I label `other` and ignore.

In [14]:
from Bio import SeqIO
from Bio.Seq import Seq
CORE = 'AATCACGTGGGCCCC'                           # common 15-bp core of the two variants of the 22-bp motif
CORE_RC = str(Seq(CORE).reverse_complement())

def classify_unit(seq):
    s = str(seq).upper(); L = len(s)
    if L >= 22 and min(1, 22*(s.count(CORE) + s.count(CORE_RC))/L) >= 0.6: return 'SatIII'
    for n in range(1, 7):                          # SRF returns HORs: multiples of the monomer
        if abs(L - 816*n) <= 0.04*816*n: return 'SatI'
    for n in range(1, 7):
        if abs(L - 702*n) <= 0.04*702*n: return 'SatII'
    return 'other'

units = pd.DataFrame([dict(id=r.id, length=len(r.seq), family=classify_unit(r.seq)) for r in SeqIO.parse(SRFDIR/'srf.fa', 'fasta')])
print(units.groupby('family').length.agg(['count', 'min', 'max']))
display(units.sort_values(['family', 'length']))
with open(TAB/'satellites_SatI-III.fa', 'w') as fo:
    for rec in SeqIO.parse(SRFDIR/'srf.fa', 'fasta'):
        c = units.set_index('id').loc[rec.id, 'family']
        if c != 'other': fo.write(f'>{c}|{rec.id}\n{rec.seq}\n')
units.to_csv(TAB/'srf_units.csv', index=False)
shutil.copy(SRFDIR/'srf.fa', TAB/'srf.fa')

        count   min   max
family                   
SatI        7   816  4080
SatII       2  1406  2043
SatIII      7    44  2115
other       5     6    77


,id,length,family
9,prefix#circ10-816,816,SatI
14,prefix#circ15-2445,2445,SatI
13,prefix#circ14-2459,2459,SatI
6,prefix#circ7-3235,3235,SatI
15,prefix#circ16-3262,3262,SatI
8,prefix#circ9-4067,4067,SatI
10,prefix#circ11-4080,4080,SatI
1,prefix#circ2-1406,1406,SatII
12,prefix#circ13-2043,2043,SatII
17,prefix#circ18-44,44,SatIII


PosixPath('<repo>/data/srf.fa')

## 2.4 · Annotating the satellites
I use the SRF units (extended with `srfutils enlong`) as the minimap2 index and each proximal window as the query. I skip base-level alignment because it takes hours on few cores. Every 100-bp bin goes to the family of its best hit, so families never overlap, which is precise enough for centromeres of 0.6 to 10 Mb.

As a sanity check I compare with the paper. In T2T-goat2.0 I get about 117 Mb of SatI and 27 Mb of SatII, against 118.2 and 26.5 Mb in Wu et al. This is not a strict comparison, because my totals only cover the oriented windows and leave out the Y, while the paper gives genome-wide totals. I get only 6 Mb of SatIII because its 22-bp motif aligns poorly.

In [15]:
gaps_df = gaps.copy()      # gaps of the oriented proximal windows (updated if the orientation of any chromosome changes)

In [16]:
ANN = CLEAN/'anot'; ANN.mkdir(exist_ok=True)
def run_bash(s):
    subprocess.run(['bash','-o','pipefail','-c',s], check=True)

run_bash(f"{Q(K8)} {Q(SRFUTILS)} enlong {Q(SRFDIR/'srf.fa')} > {Q(ANN/'units_long.fa')}")
cls_of = units.set_index('id').family.to_dict()
CI = {'SatI':0, 'SatII':1, 'SatIII':2, 'other':3}; CN = {v:k for k,v in CI.items()}
BINA = 100

def annotate(fa_path, tag):
    # units = index, chromosomes = query; each 100-bp stretch is assigned to the best hit (no overlap between families)
    paf = ANN/f'{tag}.paf'; t0 = time.time()
    run_bash(f"minimap2 -N 1000000 -f 1000 -r 100,100 -t {THREADS} {Q(ANN/'units_long.fa')} {Q(fa_path)} > {Q(paf)} 2>/dev/null")
    df = pd.read_csv(paf, sep='\t', header=None, usecols=[0,1,2,3,5,9,10], names=['q','ql','qs','qe','t','nm','bl'])
    df['ci'] = df.t.map(cls_of).fillna('other').map(CI); df['idn'] = df.nm/df.bl
    df = df.sort_values('idn', ascending=False)
    rows = []
    for q, g in df.groupby('q'):
        arr = np.full(int(g.ql.iloc[0]//BINA) + 1, -1, np.int8)
        for s_, e_, c_ in zip((g.qs//BINA).to_numpy(), ((g.qe-1)//BINA + 1).to_numpy(), g.ci.to_numpy()):
            seg = arr[s_:e_]; seg[seg < 0] = c_
        cut = np.flatnonzero(np.diff(np.r_[-2, arr, -2]))
        for a_, b_ in zip(cut[:-1], cut[1:]):
            if arr[a_] >= 0:
                hap_, chr_ = q.split('_', 1)
                rows.append(dict(hap=hap_, chr=chr_, start=int(a_*BINA), end=int(b_*BINA), family=CN[int(arr[a_])], bp=int((b_-a_)*BINA)))
    print(tag, f'{len(df)} hits, {time.time()-t0:.0f} s')
    return pd.DataFrame(rows, columns=['hap','chr','start','end','family','bp'])

ann = pd.concat([annotate(PROX[h], h) for h in GENOME], ignore_index=True)

T2T 1706046 hits, 142 s


P 1456144 hits, 121 s


M 1415424 hits, 133 s


### Orientation review
The first guess in 2.2 only uses SatI, so a chromosome whose centromere has little SatI, like the X, ends up with no signal on either side. I review the chromosomes with less than 500 kb of satellite in the window I chose: I annotate their opposite end as well and switch sides if that end clearly has more satellite.

In [17]:
SAT3 = ['SatI','SatII','SatIII']
sat_tot = ann[ann.family.isin(SAT3)].groupby(['hap','chr']).bp.sum()
chk = chroms[chroms.placed & (chroms.kind != 'Y')][['hap','chr','seqname','length']]
weak = chk[[sat_tot.get((r.hap, r.chr), 0) < 500_000 for r in chk.itertuples()]]
print('Chromosomes with <500 kb of satellite in the chosen window:', weak[['hap','chr']].values.tolist())
side_now = {(r.hap, r.chr): r.centromere_side for r in orient.itertuples()}
swapped, alt_tables = [], []
for hap in weak.hap.unique():
    faG = Fasta(str(GENOME[hap]), as_raw=True, sequence_always_upper=True)
    altp = ANN/f'{hap}.alt.fa'; altseq = {}
    with open(altp, 'w') as fo:
        for r in weak[weak.hap == hap].itertuples():
            w = min(W, r.length)
            s = str(Seq(faG[r.seqname][r.length-w:r.length]).reverse_complement()) if side_now[(hap, r.chr)] == 'L' else faG[r.seqname][0:w]
            altseq[r.chr] = s; fo.write(f'>{r.seqname}\n{s}\n')
    alt = annotate(altp, f'{hap}.alt')
    alt_sat = alt[alt.family.isin(SAT3)].groupby('chr').bp.sum()
    for r in weak[weak.hap == hap].itertuples():
        a_alt, a_cur = alt_sat.get(r.chr, 0), sat_tot.get((hap, r.chr), 0)
        if a_alt >= 200_000 and a_alt > 2*a_cur:           # the other end clearly carries the centromere
            ann = pd.concat([ann[~((ann.hap == hap) & (ann.chr == r.chr))], alt[alt.chr == r.chr]], ignore_index=True)
            gaps_df = gaps_df[~((gaps_df.hap == hap) & (gaps_df.chr == r.chr))]
            gaps_df = pd.concat([gaps_df, pd.DataFrame([dict(hap=hap, chr=r.chr, kind='X' if r.chr=='CHIX' else 'autosome', start=a, end=b, len=b-a) for a, b in gap_runs(altseq[r.chr])])], ignore_index=True)
            swapped.append((hap, r.chr, int(a_cur), int(a_alt)))
print('Switched sides (hap, chromosome, satellite bp before, after):', swapped)
still = [(r.hap, r.chr) for r in weak.itertuples() if (r.hap, r.chr) not in {(s[0], s[1]) for s in swapped}]
print('Still with little satellite (short centromere or no SatI–III):', still)
for h_, c_, *_ in swapped:                               # the orientation of those chromosomes changed
    m_ = (orient.hap == h_) & (orient.chr == c_)
    orient.loc[m_, 'centromere_side'] = orient.loc[m_, 'centromere_side'].map({'L': 'R', 'R': 'L'})
orient.to_csv(TAB/'qc_orientation.csv', index=False)
gaps_df.to_csv(TAB/'proximal_gaps.csv', index=False)
ann.to_csv(TAB/'satellite_annotation.csv', index=False)
print(ann.groupby(['hap','family']).bp.sum().unstack().div(1e6).round(2))

Chromosomes with <500 kb of satellite in the chosen window: [['P', 'CHI3'], ['P', 'CHI28'], ['M', 'CHI7'], ['M', 'CHI21'], ['M', 'CHI29']]


P.alt 71734 hits, 28 s


M.alt 7241 hits, 1 s
Switched sides (hap, chromosome, satellite bp before, after): [('P', 'CHI3', 1100, 6615000)]
Still with little satellite (short centromere or no SatI–III): [('P', 'CHI28'), ('M', 'CHI7'), ('M', 'CHI21'), ('M', 'CHI29')]
family    SatI  SatII  SatIII  other
hap                                 
M       100.79  19.10    4.90   0.26
P       109.35  24.71    7.34   0.29
T2T     117.28  26.68    6.35   0.35


### Orientation check at both ends
As a last check I annotate both ends of every chromosome in the three assemblies with the SRF units. The end with more satellite should be the one I chose.

In [18]:
end_sat = {}
for hap in GENOME:
    for end in ('left', 'right'):
        a = annotate(CLEAN/f'{hap}.{end}{WINDOW_MB}Mb.fa', f'{hap}.{end}')
        a = a[a.family.isin(KINDS_SAT)].assign(chr=lambda d: d.chr.str.split(':').str[0])
        for c_, bp in a.groupby('chr').bp.sum().items(): end_sat[(hap, c_, end)] = bp/1e6
orient['sat_left_Mb']  = [end_sat.get((r.hap, r.chr, 'left'), 0) for r in orient.itertuples()]
orient['sat_right_Mb'] = [end_sat.get((r.hap, r.chr, 'right'), 0) for r in orient.itertuples()]
orient['side_by_satellite'] = np.where(orient.sat_left_Mb >= orient.sat_right_Mb, 'L', 'R')
orient['orientation_ok'] = orient.side_by_satellite.eq(orient.centromere_side)
print('Chromosomes whose chosen end is also the end with more satellite, by haplotype:'); print(orient.groupby('hap').orientation_ok.agg(['sum', 'size']))
print('End with more satellite in the deposited sequence:', orient.groupby('hap').side_by_satellite.value_counts().to_dict())
print('Mismatches:', orient[~orient.orientation_ok][['hap', 'chr', 'sat_left_Mb', 'sat_right_Mb', 'centromere_side']].round(2).values.tolist())
orient.to_csv(TAB/'qc_orientation.csv', index=False)

T2T.left 1709132 hits, 151 s


T2T.right 61786 hits, 5 s


P.left 1531065 hits, 130 s


P.right 46230 hits, 4 s


M.left 1415461 hits, 138 s


M.right 60380 hits, 5 s
Chromosomes whose chosen end is also the end with more satellite, by haplotype:
     sum  size
hap           
M     30    30
P     29    29
T2T   30    30
End with more satellite in the deposited sequence: {('M', 'L'): 30, ('P', 'L'): 29, ('T2T', 'L'): 30}
Mismatches: []


## 2.5 · Delimiting each centromere
I define the centromere as the satellite block with the most sequence in the proximal window, after merging blocks that are closer than 500 kb. Its length is its end minus its start, so it includes any non-satellite sequence between the merged pieces. I also add up the bp of each family inside the block.

Two flags go with each centromere. It is `incomplete` if there is a gap within 100 kb of the block or if the block is shorter than 250 kb. It is `low_satellite` if satellite is less than 30 % of the block (`sat_frac`). That second flag does not say that the region is not a centromere. It says that I cannot measure its length reliably, either because sequence is missing in the assembly or because the satellite is not in my library. The 30 % threshold falls in a gap in the data: the three flagged blocks, all in haplotype M, have `sat_frac` between 0.10 and 0.21, and the lowest of the rest is 0.44.

I exclude the Y. Because the block is the whole satellite domain and not the CENP-A domain, it can go beyond the range reported by Wu et al.: in T2T-goat2.0 it goes from 0.85 to 10.4 Mb, and the shortest is the X. On the X the main block (about 0.6 Mb of SatII and almost no SatI) lies 7.2 Mb from the end, which is the short arm. I check it in 2.7.

In [19]:
def centromere_block(a):
    # satellite blocks separated by < MAX_GAP; the centromere is the block with the MOST satellite (not the first one:
    # on the X there is a SatIII remnant near the end and the real block lies further in)
    a = a.sort_values('start'); blocks, cur = [], None
    for r in a.itertuples():
        if cur is None: cur = [r.start, r.end, r.bp]
        elif r.start - cur[1] <= MAX_GAP: cur[1] = max(cur[1], r.end); cur[2] += r.bp
        else: blocks.append(cur); cur = [r.start, r.end, r.bp]
    blocks.append(cur); b = max(blocks, key=lambda x: x[2])
    return int(b[0]), int(b[1])

rows = []
sat_ann = ann[ann.family.isin(KINDS_SAT)]
for r in chroms[chroms.placed & (chroms.kind != 'Y')].itertuples():
    a = sat_ann[(sat_ann.hap == r.hap) & (sat_ann.chr == r.chr)]
    if a.empty:
        rows.append(dict(hap=r.hap, chr=r.chr, kind=r.kind, length=r.length, cen_start=np.nan, cen_end=np.nan, cen_len=np.nan)); continue
    s, e = centromere_block(a)
    inside = a[(a.start >= s) & (a.end <= e)]
    d = dict(hap=r.hap, chr=r.chr, kind=r.kind, length=r.length, cen_start=s, cen_end=e, cen_len=e - s)
    d.update({k: int(inside[inside.family == k].bp.sum()) for k in KINDS_SAT})
    g = gaps_df[(gaps_df.hap == r.hap) & (gaps_df.chr == r.chr)]
    d['n_gaps_cen'] = int(((g.start < e + 100_000) & (g.end > max(0, s - 100_000))).sum()) if len(g) else 0
    rows.append(d)
cen = pd.DataFrame(rows)
cen['complete'] = cen.n_gaps_cen.eq(0) & cen.cen_len.notna() & (cen.cen_len >= 250_000)   # <250 kb = centromere not detected
cen['sat_bp'] = cen[KINDS_SAT].sum(axis=1)
cen['sat_frac'] = cen.sat_bp/cen.cen_len
cen['low_satellite'] = cen.sat_frac < MIN_SAT_FRAC     # length not reliable: little satellite in the block
cen.to_csv(TAB/'centromeres.csv', index=False)

T = cen[cen.hap == 'T2T']
print('T2T-goat2.0 · centromere length (Mb): min %.2f (%s) · max %.2f (%s) · median %.2f' % (
      T.cen_len.min()/1e6, T.loc[T.cen_len.idxmin(),'chr'], T.cen_len.max()/1e6, T.loc[T.cen_len.idxmax(),'chr'], T.cen_len.median()/1e6))
print('Expected (Wu et al. 2024): 0.6–8.51 Mb, shortest on X. SatI ≈ 67.81 % of the centromeres.')
print('SatI / sum of centromeric blocks: %.1f %%' % (100*T.SatI.sum()/T.cen_len.sum()))
print('Incomplete centromeres by haplotype:', cen.groupby('hap').complete.apply(lambda s: int((~s).sum())).to_dict(), '(paper: ~9 in P, ~6 in M)')
print('Flagged low_satellite:', cen[cen.low_satellite][['hap','chr']].values.tolist(), '| sat_frac of the rest, minimum: %.2f' % cen[~cen.low_satellite].sat_frac.min())
cen[cen.hap=='T2T'].head()

T2T-goat2.0 · centromere length (Mb): min 0.85 (CHIX) · max 10.38 (CHI16) · median 7.14
Expected (Wu et al. 2024): 0.6–8.51 Mb, shortest on X. SatI ≈ 67.81 % of the centromeres.
SatI / sum of centromeric blocks: 55.5 %
Incomplete centromeres by haplotype: {'M': 5, 'P': 9, 'T2T': 0} (paper: ~9 in P, ~6 in M)
Flagged low_satellite: [['M', 'CHI7'], ['M', 'CHI9'], ['M', 'CHI29']] | sat_frac of the rest, minimum: 0.44


,hap,chr,kind,length,cen_start,cen_end,cen_len,SatI,SatII,SatIII,n_gaps_cen,complete,sat_bp,sat_frac,low_satellite
0,T2T,CHI1,autosome,166528736,27200,9191300,9164100,5727200,970100,365600,0,True,7062900,0.770714,False
1,T2T,CHI2,autosome,147555794,31600,10257700,10226100,6489800,1049500,330800,0,True,7870100,0.769609,False
2,T2T,CHI3,autosome,132669774,33300,8881400,8848100,4763300,863600,602900,0,True,6229800,0.704083,False
3,T2T,CHI4,autosome,126865584,28400,5185600,5157200,2918100,1286500,0,0,True,4204600,0.815287,False
4,T2T,CHI5,autosome,127915634,28900,7786700,7757800,2713700,963000,253400,0,True,3930100,0.506600,False


## 2.6 · Master table
I keep one row per chromosome and haplotype, with everything notebook 03 needs.

| Column | Meaning |
|---|---|
| `hap`, `chr`, `kind` | haplotype (T2T, P, M), chromosome (`CHI1`…`CHIX`) and type (autosome or X) |
| `length` | chromosome length (bp) |
| `cen_start`, `cen_end`, `cen_len` | start, end and length of the centromeric block in the oriented window (bp) |
| `SatI`, `SatII`, `SatIII`, `sat_bp` | bp of each family inside the block, and the total |
| `sat_frac`, `low_satellite` | satellite as a fraction of the block, and the flag for less than 30 % |
| `n_gaps_cen`, `complete` | gaps near the centromere, and whether it counts as complete |
| `centromere_side` | end of the deposited sequence where the centromere lies (`L` or `R`) |
| `cen_Mb`, `sat_Mb`, `rest_Mb` | centromere, satellite and rest of the chromosome (Mb) |
| `prop_SatI`…`prop_SatIII` | share of each family in the satellite of the block |

In [20]:
master = cen.merge(orient[['hap', 'chr', 'centromere_side']], on=['hap', 'chr'], how='left')
master['cen_Mb']  = master.cen_len/1e6
master['sat_Mb']  = master.sat_bp/1e6
master['rest_Mb'] = (master.length - master.cen_len)/1e6
for k in KINDS_SAT: master['prop_'+k] = master[k]/master.sat_bp.replace(0, np.nan)
master.to_csv(TAB/'centromeres_master.csv', index=False)
print(master.shape, '→', TAB/'centromeres_master.csv')
master[master.hap == 'T2T'].round(3).head()

(89, 22) → <repo>/data/centromeres_master.csv


,hap,chr,kind,length,cen_start,cen_end,cen_len,SatI,SatII,SatIII,...,sat_bp,sat_frac,low_satellite,centromere_side,cen_Mb,sat_Mb,rest_Mb,prop_SatI,prop_SatII,prop_SatIII
0,T2T,CHI1,autosome,166528736,27200,9191300,9164100,5727200,970100,365600,...,7062900,0.771,False,L,9.164,7.063,157.365,0.811,0.137,0.052
1,T2T,CHI2,autosome,147555794,31600,10257700,10226100,6489800,1049500,330800,...,7870100,0.770,False,L,10.226,7.870,137.330,0.825,0.133,0.042
2,T2T,CHI3,autosome,132669774,33300,8881400,8848100,4763300,863600,602900,...,6229800,0.704,False,L,8.848,6.230,123.822,0.765,0.139,0.097
3,T2T,CHI4,autosome,126865584,28400,5185600,5157200,2918100,1286500,0,...,4204600,0.815,False,L,5.157,4.205,121.708,0.694,0.306,0.000
4,T2T,CHI5,autosome,127915634,28900,7786700,7757800,2713700,963000,253400,...,3930100,0.507,False,L,7.758,3.930,120.158,0.690,0.245,0.064


## 2.7 · The X chromosome
The X has the shortest centromere (about 0.85 Mb), and its only satellite block lies 7.2 Mb from the end, not 30 kb as in the autosomes. Before I use it, I want to know whether this is real or a detection, orientation or assembly error.

In the literature the goat X is acrocentric (Hansen 2009; Kowal-Mierzwa et al. 2026; Wu et al. 2024), although Putri et al. 2018 measured it as subtelocentric. Wu et al. describe a pseudoautosomal region (PAR) of about 7 Mb on the short arms of the X and the Y. I ran four checks on T2T-goat2.0: whether the first 7 Mb of the X match the Y, whether the short arm has a gene density similar to the rest of the X, whether there are other satellite blocks on the X (and whether SRF finds new units when I run it on the X alone), and whether the X of haplotype M lines up with the X of T2T-goat2.0.

In [21]:
XDIR = CLEAN/'xcheck'; XDIR.mkdir(exist_ok=True)
def faidx(genome, name, out): subprocess.run(f"samtools faidx {Q(genome)} {name} > {Q(out)}", shell=True, check=True)
faidx(GENOME['T2T'], 'T2T_CHIX', XDIR/'X.fa'); faidx(GENOME['T2T'], 'T2T_CHIY', XDIR/'Y.fa'); faidx(GENOME['M'], 'M_CHIX', XDIR/'MX.fa')
paf_cols = ['q', 'ql', 'qs', 'qe', 'strand', 't', 'tl', 'ts', 'te', 'nm', 'al', 'mq']
def paf_table(args, out):
    run_bash(f"minimap2 {args} -t {THREADS} > {Q(out)} 2>/dev/null")
    return pd.read_csv(out, sep='\t', header=None, usecols=range(12), names=paf_cols)

subprocess.run(['samtools', 'faidx', str(XDIR/'X.fa')], check=True)
xlen = int(open(str(XDIR/'X.fa') + '.fai').read().split('\t')[1])
# 1) PAR: the longest alignment of the X against the Y
xy = paf_table(f"-x asm10 {Q(XDIR/'Y.fa')} {Q(XDIR/'X.fa')}", XDIR/'X_vs_Y.paf'); par = xy.sort_values('al', ascending=False).iloc[0]
print(f'PAR: X {par.qs/1e6:.3f}–{par.qe/1e6:.3f} Mb  ↔  Y {par.ts/1e6:.3f}–{par.te/1e6:.3f} Mb · strand {par.strand} · identity ≈ {par.nm/par.al:.2f} · {par.al/1e6:.2f} Mb')
# 4) collinearity of the X of haplotype M with the X of T2T
mx = paf_table(f"-x asm5 {Q(XDIR/'X.fa')} {Q(XDIR/'MX.fa')}", XDIR/'MX_vs_X.paf'); mb = mx.sort_values('al', ascending=False).iloc[0]
print(f'X(M) vs X(T2T): main block {mb.al/1e6:.2f} Mb of {mb.ql/1e6:.2f} Mb, strand {mb.strand}, coordinates {mb.qs/1e6:.2f}–{mb.qe/1e6:.2f} ↔ {mb.ts/1e6:.2f}–{mb.te/1e6:.2f} Mb')

# 2) genes on the X (NCBI annotation)
gff = RAW/'GCA_041735815.1_T2T-goat2.0_genomic.gff.gz'
if not gff.exists():
    fetch(ncbi_dir(*ASSEMBLIES['T2T']) + 'GCA_041735815.1_T2T-goat2.0_genomic.gff.gz', gff)
xacc = chroms[(chroms.hap == 'T2T') & (chroms.chr == 'CHIX')].accession.iloc[0]
gm = []
with gzip.open(gff, 'rt') as f:
    for l in f:
        c = l.rstrip('\n').split('\t')
        if len(c) > 8 and c[0] == xacc and c[2] == 'gene': gm.append((int(c[3]) + int(c[4]))/2e6)
gm = np.array(gm); xmb = xlen/1e6
print(f'Genes on the X: {len(gm)} ({len(gm)/xmb:.1f}/Mb) · in the first 7 Mb (short arm): {int((gm < 7).sum())} ({(gm < 7).sum()/7:.1f}/Mb)')

PAR: X 0.012–7.040 Mb  ↔  Y 0.019–7.020 Mb · strand + · identity ≈ 0.89 · 7.07 Mb


X(M) vs X(T2T): main block 144.68 Mb of 144.69 Mb, strand +, coordinates 0.01–144.69 ↔ 0.01–144.69 Mb


Genes on the X: 1226 (8.5/Mb) · in the first 7 Mb (short arm): 62 (8.9/Mb)


In [22]:
# 3) the whole X: satellite blocks and SRF applied to the X alone
axs = annotate(XDIR/'X.fa', 'X_full'); axs = axs[axs.family.isin(KINDS_SAT)].sort_values('start')
blocks, cur = [], None
for r in axs.itertuples():
    if cur is not None and r.start - cur[1] <= 20_000: cur[1] = max(cur[1], r.end); cur[2] += r.bp
    else:
        if cur: blocks.append(cur)
        cur = [r.start, r.end, r.bp]
blocks.append(cur); blocks = [b for b in blocks if b[1] - b[0] >= 30_000]
print('Satellite blocks ≥ 30 kb along the whole X (Mb):', [(round(a/1e6, 3), round(b/1e6, 3), f'{n/1e3:.0f} kb of satellite') for a, b, n in blocks])
fam = axs.groupby('family').bp.sum().to_dict(); print('Satellite on the whole X (bp):', fam)

SX = XDIR/'srf'; SX.mkdir(exist_ok=True); subprocess.run(f"rm -rf {Q(SX)}/tmp; mkdir -p {Q(SX)}/tmp", shell=True, check=True)
run_bash(f"cd {Q(SX)} && {Q(KMC)} -fm -k151 -t{min(THREADS, 8)} -m{KMC_RAM_GB} -sm -ci20 -cs1000000 {Q(XDIR/'X.fa')} count.kmc tmp > /dev/null && {Q(KMC_DUMP)} count.kmc count.txt && {Q(SRF)} -p xs count.txt > srf_X.fa 2> /dev/null")
ux = [(r.id, len(r.seq)) for r in SeqIO.parse(SX/'srf_X.fa', 'fasta')]
print('Units that SRF finds when analysing the X alone:', ux)

blk = max(blocks, key=lambda b: b[2]); p_arm = blk[0]
check_x = pd.DataFrame([dict(
    X_length_bp=xlen, block_start_bp=blk[0], block_end_bp=blk[1], block_satellite_bp=int(blk[2]), fraction_from_end=blk[0]/xlen,
    satellite_blocks_ge30kb=len(blocks), SRF_units_on_X=len(ux), SatII_bp=int(fam.get('SatII', 0)), SatI_bp=int(fam.get('SatI', 0)), SatIII_bp=int(fam.get('SatIII', 0)),
    PAR_X_start=int(par.qs), PAR_X_end=int(par.qe), PAR_Y_start=int(par.ts), PAR_Y_end=int(par.te), PAR_identity=par.nm/par.al,
    genes_X=len(gm), genes_per_Mb_X=len(gm)/xmb, genes_per_Mb_first7Mb=(gm < 7).sum()/7,
    X_haplotypeM_collinear_bp=int(mb.al), X_haplotypeM_strand=mb.strand)])
check_x.to_csv(TAB/'x_chromosome_check.csv', index=False); display(check_x.T)
print(f'Reading: the only satellite block of the X is {blk[0]/1e6:.2f} Mb from the end ({100*blk[0]/xlen:.1f} % of its length); '
      f'the ~{par.qe/1e6:.1f} Mb short arm is homologous to the Y (PAR) and has a gene density similar to the rest of the X. There is no evidence of a detection or orientation error.')

X_full 3224 hits, 3 s
Satellite blocks ≥ 30 kb along the whole X (Mb): [(7.233, 8.085, '638 kb of satellite')]
Satellite on the whole X (bp): {'SatI': 46800, 'SatII': 592100}


**********
Stage 1: 100%


Stage 2: 100%


Units that SRF finds when analysing the X alone: [('xs#circ1-6', 6), ('xs#circ2-2801', 2801), ('xs#circ3-17', 17), ('xs#circ4-50', 50)]


,0
X_length_bp,144688805
block_start_bp,7232600
block_end_bp,8085400
block_satellite_bp,638500
fraction_from_end,0.049987
satellite_blocks_ge30kb,1
SRF_units_on_X,4
SatII_bp,592100
SatI_bp,46800
SatIII_bp,0


Reading: the only satellite block of the X is 7.23 Mb from the end (5.0 % of its length); the ~7.0 Mb short arm is homologous to the Y (PAR) and has a gene density similar to the rest of the X. There is no evidence of a detection or orientation error.


### Files written to `data/`
| File | Content |
|---|---|
| `chromosomes.csv`, `manifest.csv`, `public_runs.csv` | chromosome lengths by haplotype, download traceability and the PacBio catalogue |
| `srf_units.csv`, `satellites_SatI-III.fa`, `srf.fa` | satellite units and their classification |
| `satellite_annotation.csv` | SatI–III intervals by chromosome and haplotype |
| `centromeres.csv`, `centromeres_master.csv` | centromeres by chromosome and haplotype (the master table adds derived columns) |
| `qc_*.csv`, `proximal_gaps.csv` | quality controls: orientation, windows, gaps |
| `x_chromosome_check.csv` | evidence on the position of the X centromere |